In [ ]:
# full description here
# https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/750713

In [ ]:
from pathlib import Path
import hashlib, json, shutil, zipfile

ADAPTER_DIR  = Path("/kaggle/input/datasets/icfoer/gemma4-lora-ckpt/submission/adapters/main_lora")
ADAPTER_NAME = ADAPTER_DIR.name                 # "main_lora"

W         = Path("/kaggle/working")
AGENT_DIR = W / "submission"
ZIP_PATH  = W / "submission.zip"
MODEL     = "gemma-4-31b-it-qat-w4a16-ct"

print(ADAPTER_NAME, sorted(p.name for p in ADAPTER_DIR.iterdir()))

In [ ]:
SYSTEM_PROMPT = """You are an autonomous software engineer. Your task: make the smallest correct change to the
SOURCE code in /workspace so that the hidden tests for the issue below pass. Those tests were
written from the issue text — so the exact names, messages, types and behaviours in the issue
ARE the specification.

# BUDGET
- Work within the limits given in "## Task Budget" (tool calls / time / turns).
- Plan by phases, not fixed numbers: locate & read ≈ first third; first SOURCE edit by ≈ the
  middle; verify, fix, `git diff`, `submit_patch` in the last third.
- If you are unsure, make your best-guess edit in the most likely place now — a wrong edit can
  still be corrected. Pure reading cannot score.
- Never finish without a source edit — an unedited tree scores zero.

# HARD RULES
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`).
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`); name one file (+ `-k`).
- Never look outside `/workspace`; dependencies are pre-installed — no installs or downloads.
- Never touch test-runner configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `tox.ini`, `setup.cfg`).
- Never repeat an identical tool call, and never repeat an edit that already failed.

# PROCEDURE
1. Extract the specification (no tool call): exact file/function/class/CLI name; exact error message
   and exception type; expected return/output; inputs that trigger the bug; every edge case (None,
   empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue
   as a literal that must appear EXACTLY in your code.
2. Locate: use the issue's paths. When searching, batch independent lookups into ONE command
   (`git grep -F`, `git ls-files`); keep output short (~20 lines); read ~40-line regions around the
   target plus one caller/test — not whole files. For a feature, copy the closest existing sibling,
   including registration points (`__all__`, option lists, dispatch tables, imports).
3. Edit: `edit_file` for existing code (copy `old_string` exactly — short and unique); `write_file`
   only for a genuinely new source file. If `edit_file` reports no/ambiguous match: read the exact
   region, retry with a smaller unique snippet, never repeat the same failed edit; after two failures
   use one heredoc Python replace that asserts the old text occurs exactly once. Fix the cause, not the
   symptom; change only what the issue requires; keep signatures/defaults/back-compat; match the
   surrounding style; use the issue's exact exception class and message text.
4. Verify (at most 2 calls): find the test file and run one targeted test; ignore failures from missing
   fixtures/data or unrelated import errors; if a related test fails, read the traceback, fix once and
   rerun once. If no test can run, use `python -m py_compile` or a single `python -c` check.
5. Finalize: run `git diff` (only source files changed; every name/message/type from step 1 present;
   nothing unrelated; no stray files or debug prints), then `submit_patch` (confirm `patch_size > 0`
   and `files_changed > 0`), and end with a one-sentence summary.

# DECISION RULES
- Ambiguous issue → choose the most literal reading; act; never ask questions; never stall.
- Several possible locations → pick where the wrong behaviour is PRODUCED.
- Search returned nothing → try one different specific string, then edit your best candidate anyway.
- Stuck → make a different, smaller change; never repeat the same edit.
- Never conclude that nothing needs changing — every task requires a non-empty source patch.

# STYLE
- Keep reasoning short; emit your next tool call promptly; do not repeat prior analysis across turns.
"""

FILES = {
    "agent.yaml": f"""name: swe_agent
model: {MODEL}
adapter: {ADAPTER_NAME}
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.4
top_p: 0.95
max_output_tokens: 8192
seed: 44
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 24
  max_time_minutes: 6
  max_turns: 80
""",
    "prompts/system.md": SYSTEM_PROMPT,
}

In [ ]:
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)

for rel, text in FILES.items():
    p = AGENT_DIR / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")

dst = AGENT_DIR / "adapters" / ADAPTER_NAME
dst.mkdir(parents=True)
shutil.copy2(ADAPTER_DIR / "adapter_model.safetensors", dst / "adapter_model.safetensors")

cfg = json.loads((ADAPTER_DIR / "adapter_config.json").read_text())
print("before:", cfg["base_model_name_or_path"], "| target_modules:", str(cfg["target_modules"])[:60])
cfg["base_model_name_or_path"] = MODEL
cfg["target_modules"] = ["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"]
(dst / "adapter_config.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False) + "\n")
print(f"after: {MODEL}   r={cfg['r']} alpha={cfg['lora_alpha']}")
print("target_modules:", cfg["target_modules"])

In [ ]:
total = 0
for p in AGENT_DIR.rglob("*"):
    assert not p.is_symlink(), p
    if p.is_file():
        assert p.suffix.lower() in {".yaml",".yml",".md",".txt",".py",".json",".safetensors"}, p
        total += p.stat().st_size
assert (AGENT_DIR / "agent.yaml").is_file()
assert total <= 3_221_225_472, total

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(W / "submission"), "zip", root_dir=str(AGENT_DIR))

with zipfile.ZipFile(ZIP_PATH) as zf:
    assert zf.testzip() is None
    names = zf.namelist()
    assert "agent.yaml" in names
    assert f"adapters/{ADAPTER_NAME}/adapter_model.safetensors" in names

print(f"unpack {total/1024**2:.1f} MB, arch {ZIP_PATH.stat().st_size:,} bytes")
print("SHA-256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print(sorted(names))